# Upsampling Visualization
Compare class distributions before and after upsampling.

In [1]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), "../.."))
if not os.path.exists(os.path.join(PROJECT_ROOT, "data", "raw")):
    PROJECT_ROOT = r"C:\Users\edlun\Desktop\DTU\Bachelor\codLLM"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from codllm.config import Config
from codllm.data_handler import (
    DataHandler,
    select_floor_upsample_targets,
    upsample,
)

## Floor upsampling

One parameter: **floor** — every class with fewer than `floor` samples gets padded up to `floor`. Everything else stays untouched. Original class ordering is preserved.

In [2]:
TRAINING_INPUT = "cod"
DATASET_SIZE = 1.0
SEED = 42

cfg = Config(
    training_input=[TRAINING_INPUT],
    dataset_size=DATASET_SIZE,
    seed=SEED,
    balance_strategy="none",
)
handler = DataHandler(cfg)
splits = handler.get_splits()
train_df = splits.train.copy()

label_col = cfg.dataset_label_column
before_counts = train_df[label_col].value_counts().sort_values(ascending=False)

print(f"Train samples: {len(train_df)}")
print(f"Unique classes: {before_counts.nunique()}")
print(f"Min class count: {before_counts.min()}")
print(f"Max class count: {before_counts.max()}")
print(f"Median class count: {before_counts.median()}")

Train samples: 908129
Unique classes: 432
Min class count: 1
Max class count: 43346
Median class count: 6.0


In [3]:
%matplotlib inline
from ipywidgets import interact, IntSlider, FloatSlider
import matplotlib.ticker as ticker

all_labels = before_counts.index.tolist()
before_vals = before_counts.reindex(all_labels).fillna(0).values
x = np.arange(len(all_labels))

def plot_floor(floor=10, decay=0.1, log_base=10.0):
    tc = select_floor_upsample_targets(
        df=train_df, label_column=label_col,
        floor=floor, decay=decay,
    )
    up_df = upsample(df=train_df, label_column=label_col, target_counts=tc, seed=42)
    after = up_df[label_col].value_counts()
    after_vals = after.reindex(all_labels).fillna(0).values
    synthetic_vals = np.clip(after_vals - before_vals, 0, None)
    n_synthetic = int(synthetic_vals.sum())
    n_affected = sum(1 for v in tc.values() if v > 0)

    affected_mask = synthetic_vals > 0
    affected_before = before_vals[affected_mask]
    affected_synth = synthetic_vals[affected_mask]
    affected_x = np.arange(affected_mask.sum())

    fig, axes = plt.subplots(1, 3, figsize=(20, 4))

    # Left: full distribution linear
    axes[0].bar(x, before_vals, width=1.0, color="steelblue", edgecolor="none", label="Original")
    axes[0].bar(x, synthetic_vals, width=1.0, bottom=before_vals, color="limegreen", edgecolor="none", label=f"Synthetic (+{n_synthetic:,})")
    axes[0].axhline(y=floor, color="red", ls="--", lw=1, label=f"floor = {floor}")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"All classes  |  {len(up_df):,} samples")
    axes[0].legend(fontsize=7)
    axes[0].set_xticks([])

    # Middle: full distribution log scale with tunable base
    axes[1].bar(x, before_vals, width=1.0, color="steelblue", edgecolor="none", label="Original")
    axes[1].bar(x, synthetic_vals, width=1.0, bottom=before_vals, color="limegreen", edgecolor="none", label=f"Synthetic (+{n_synthetic:,})")
    axes[1].axhline(y=floor, color="red", ls="--", lw=1, label=f"floor = {floor}")
    axes[1].set_yscale("log", base=log_base)
    axes[1].set_ylabel(f"Count (log{log_base:.0f})")
    axes[1].set_title("Log scale")
    axes[1].legend(fontsize=7)
    axes[1].set_xticks([])

    # Right: only affected classes
    if affected_mask.sum() > 0:
        axes[2].bar(affected_x, affected_before, width=1.0, color="steelblue", edgecolor="none", label="Original")
        axes[2].bar(affected_x, affected_synth, width=1.0, bottom=affected_before, color="limegreen", edgecolor="none", label="Synthetic")
        axes[2].axhline(y=floor, color="red", ls="--", lw=1, label=f"floor = {floor}")
        axes[2].set_ylabel("Count")
        axes[2].set_title(f"{n_affected} boosted classes")
        axes[2].legend(fontsize=7)
        axes[2].set_xticks([])
    else:
        axes[2].text(0.5, 0.5, "No classes below floor", ha="center", va="center", transform=axes[2].transAxes)

    fig.suptitle(f"floor={floor}  |  decay={decay:.0%}", fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

interact(
    plot_floor,
    floor=IntSlider(value=10, min=1, max=200, step=1, description="Floor"),
    decay=FloatSlider(value=0.1, min=0.0, max=0.5, step=0.05, description="Decay"),
    log_base=FloatSlider(value=10.0, min=2.0, max=50.0, step=1.0, description="Log Base"),
);

interactive(children=(IntSlider(value=10, description='Floor', max=200, min=1), FloatSlider(value=0.1, descrip…